# LeetCode 2351 — First Letter to Appear Twice bằng Python

Link đề: https://leetcode.com/problems/first-letter-to-appear-twice/ · Độ khó: **Easy** · Chủ đề: Hash Table, String, Counting

Skeleton LeetCode:

```python
class Solution:
    def repeatedCharacter(self, s: str) -> str:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Phân biệt rõ **"chữ cái đầu tiên lặp lại"** (xét theo vị trí lần xuất hiện **thứ hai**) với "chữ cái xuất hiện đầu tiên trong chuỗi" hoặc "chữ cái xuất hiện nhiều nhất".
2. Dùng `set` để ghi nhớ "những gì đã thấy" trong một lượt duyệt, với kiểm tra `x in seen` trung bình `O(1)`.
3. Biết các cách thay `set`: mảng boolean 26 phần tử, **bitmask** một số nguyên, và `str.find` để tìm vị trí lần thứ hai.
4. Viết 5 cách: lục lại phần đầu chuỗi (brute force), `set`, mảng boolean, bitmask, và tìm chỉ số lần thứ hai nhỏ nhất; đối chiếu bằng test ngẫu nhiên.
5. Tránh các lỗi: trả chữ cái có **lần xuất hiện đầu** sớm nhất thay vì lần **thứ hai** sớm nhất, thêm vào `seen` trước khi kiểm tra, dùng `Counter` rồi lấy chữ cái nhiều nhất.

## 2. Đề bài & ví dụ

Cho chuỗi `s` gồm chữ cái tiếng Anh thường. Trả về **chữ cái đầu tiên xuất hiện lần thứ hai** khi duyệt chuỗi từ trái sang phải; tức chữ cái có **lần xuất hiện thứ hai** ở chỉ số **nhỏ nhất**.

Đề đảm bảo `s` có **ít nhất một** chữ cái xuất hiện **từ hai lần trở lên**.

| Ví dụ | s | Output | Giải thích |
|---|---|---|---|
| 1 | `"abccbaacz"` | `"c"` | lần thứ hai của `c` ở chỉ số 3; `b` ở chỉ số 4, `a` ở chỉ số 5, nên `c` đến trước |
| 2 | `"abcdd"` | `"d"` | chỉ `d` lặp lại |

Constraints: `2 <= len(s) <= 100`; `s` gồm chữ cái tiếng Anh thường; `s` có ít nhất một chữ cái xuất hiện hai lần.

Chú ý phân biệt: trong `"abccbaacz"`, chữ `a` xuất hiện **lần đầu** sớm nhất (chỉ số 0), nhưng đáp án là `c` vì ta xét **lần xuất hiện thứ hai**.

In [1]:
examples = [("abccbaacz", "c"), ("abcdd", "d")]
for s, expected in examples:
    first = {}
    second = {}
    for i, ch in enumerate(s):
        if ch not in first:
            first[ch] = i
        elif ch not in second:
            second[ch] = i
    print(f"s={s!r}: lần đầu={first}")
    print(f"           lần hai={second} -> chữ có lần hai sớm nhất: {min(second, key=second.get)!r} ; mong đợi {expected!r}")

s='abccbaacz': lần đầu={'a': 0, 'b': 1, 'c': 2, 'z': 8}
           lần hai={'c': 3, 'b': 4, 'a': 5} -> chữ có lần hai sớm nhất: 'c' ; mong đợi 'c'
s='abcdd': lần đầu={'a': 0, 'b': 1, 'c': 2, 'd': 3}
           lần hai={'d': 4} -> chữ có lần hai sớm nhất: 'd' ; mong đợi 'd'


## 3. Kiến thức nền

### 3.1. Duyệt chuỗi với `enumerate`

`enumerate(s)` cho cặp `(chỉ số, ký tự)`. Rất hay khi cần biết **vị trí** đang xét.

In [2]:
for i, ch in enumerate("abca"):
    print(i, ch)

0 a
1 b
2 c
3 a


### 3.2. `set` và kiểm tra thành viên

`seen = set()`; `seen.add(ch)`; `ch in seen` kiểm tra có chưa (trung bình `O(1)`). `set` không lưu thứ tự và không có phần tử trùng.

In [3]:
seen = set()
for ch in "abca":
    print(ch, "đã thấy?", ch in seen)
    seen.add(ch)
print(seen)

a đã thấy? False
b đã thấy? False
c đã thấy? False
a đã thấy? True
{'b', 'c', 'a'}


### 3.3. Cắt chuỗi `s[:j]` và toán tử `in` trên chuỗi

`s[:j]` là phần của `s` **trước** chỉ số `j`. `ch in s[:j]` kiểm tra `ch` đã xuất hiện trong phần đó chưa (quét `O(j)`). Hữu ích cho cách brute force.

In [4]:
s = "abccba"
print(s[:3], "c" in s[:3], "c" in s[:2])
print(s[:0] == "")

abc True False
True


### 3.4. `str.find(sub, start)`

`s.find(ch)` trả chỉ số **lần đầu** của `ch` (hoặc `-1`). `s.find(ch, start)` tìm từ chỉ số `start` trở đi. Vì vậy lần thứ hai: `s.find(ch, s.find(ch) + 1)`.

In [5]:
s = "abccbaacz"
first_c = s.find("c")
second_c = s.find("c", first_c + 1)
print(first_c, second_c, s.find("z"), s.find("z", 9))

2 3 8 -1


### 3.5. Mảng boolean 26 phần tử và `ord`

`ord(ch) - ord("a")` đổi chữ thường thành chỉ số `0..25`. Mảng `seen = [False] * 26` thay `set` khi bảng chữ cái nhỏ.

In [6]:
print(ord("a"), ord("z"), ord("c") - ord("a"))
seen = [False] * 26
seen[ord("c") - 97] = True
print(seen[:5], sum(seen))

97 122 2
[False, False, True, False, False] 1


### 3.6. Bitmask: một số nguyên làm tập hợp

Bit thứ `k` của số nguyên `mask` là `1` ⇔ chữ cái thứ `k` đã thấy. `mask >> k & 1` kiểm tra; `mask |= 1 << k` thêm. Toàn bộ tập 26 chữ chỉ cần **một số nguyên 26 bit**.

In [7]:
mask = 0
for ch in "abca":
    k = ord(ch) - 97
    print(ch, "đã thấy?", bool(mask >> k & 1))
    mask |= 1 << k
print(bin(mask))

a đã thấy? False
b đã thấy? False
c đã thấy? False
a đã thấy? True
0b111


### 3.7. `min(..., key=...)`

`min(d, key=d.get)` trả **khóa** có giá trị nhỏ nhất trong từ điển `d`. Ta dùng nó để chọn chữ cái có chỉ số lần thứ hai nhỏ nhất.

In [8]:
second = {"a": 5, "b": 4, "c": 3}
print(min(second, key=second.get), min(second.values()))

c 3


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Tưởng tượng bạn **đọc chuỗi từng chữ cái**, và ghi một danh sách "đã gặp". Đến chữ nào mà **đã có trong danh sách**, đó chính là lần xuất hiện thứ hai của nó — và vì ta đọc từ trái sang phải, **chữ đầu tiên như vậy** là đáp án.

Ví dụ `s = "abccbaacz"`:

| chỉ số | chữ | đã gặp trước đó? | danh sách đã gặp sau bước này |
|---|---|---|---|
| 0 | `a` | không | `{a}` |
| 1 | `b` | không | `{a,b}` |
| 2 | `c` | không | `{a,b,c}` |
| 3 | `c` | **có** | → trả về `c` |

Dừng ngay: không cần đọc tiếp.

In [9]:
s = "abccbaacz"
seen = set()
for i, ch in enumerate(s):
    if ch in seen:
        print(f"i={i}: {ch!r} đã có trong {sorted(seen)} -> đáp án")
        break
    seen.add(ch)
    print(f"i={i}: {ch!r} chưa thấy -> thêm, seen={sorted(seen)}")

i=0: 'a' chưa thấy -> thêm, seen=['a']
i=1: 'b' chưa thấy -> thêm, seen=['a', 'b']
i=2: 'c' chưa thấy -> thêm, seen=['a', 'b', 'c']
i=3: 'c' đã có trong ['a', 'b', 'c'] -> đáp án


## 5. Cách 1 — Brute Force / baseline: nhìn lại phần đầu chuỗi

Với mỗi chỉ số `j` từ trái sang phải, kiểm tra `s[j]` có xuất hiện trong `s[:j]` (phần đứng trước) không. Chỉ số `j` đầu tiên thỏa là vị trí lần xuất hiện thứ hai sớm nhất.

In [10]:
def first_twice_brute(s):
    for j in range(len(s)):
        if s[j] in s[:j]:
            return s[j]
    return ""

print(first_twice_brute("abccbaacz"), first_twice_brute("abcdd"))

c d


In [11]:
from typing import List

class SolutionBrute:
    def repeatedCharacter(self, s: str) -> str:
        for j in range(len(s)):
            if s[j] in s[:j]:
                return s[j]
        return ""

print(SolutionBrute().repeatedCharacter("abccbaacz"))
print(SolutionBrute().repeatedCharacter("abcdd"))

c
d


**Độ phức tạp:** `n = len(s)`. Mỗi `s[:j]` và `in` tốn `O(j)` → tổng `O(n²)` → **Time `O(n²)`**, **Space `O(n)`** (cắt chuỗi tạo bản sao tạm).

**Nút thắt:** mỗi lần ta **quét lại** phần đầu. Chỉ cần một cấu trúc **nhớ nhanh** những gì đã thấy.

## 6. Ý tưởng thuật toán / bước đột phá

- Lưu những chữ cái **đã thấy** vào `set` (hoặc mảng/bitmask). Kiểm tra thành viên `O(1)`.
- Duyệt từ trái sang phải. Với mỗi chữ `ch`: **nếu `ch` đã trong `seen` → đây là lần thứ hai → trả về ngay**; nếu chưa → thêm vào `seen`.
- Thứ tự kiểm tra quan trọng: **kiểm tra trước, thêm sau**. Nếu thêm trước thì chữ nào cũng "đã thấy".
- Vì ta trả về ở **lần thứ hai đầu tiên**, kết quả chắc chắn là chữ có lần thứ hai ở chỉ số nhỏ nhất. Đề đảm bảo luôn có đáp án.

## 7. Các cách tiếp theo

### Cách 2 — `set` các chữ đã thấy (khuyến nghị khi nộp)

In [12]:
from typing import List

class SolutionSet:
    def repeatedCharacter(self, s: str) -> str:
        seen = set()
        for ch in s:
            if ch in seen:
                return ch
            seen.add(ch)
        return ""

print(SolutionSet().repeatedCharacter("abccbaacz"))
print(SolutionSet().repeatedCharacter("abcdd"))

c
d


**Độ phức tạp:** Time `O(n)` (thực tế ≤ 27 bước vì sau 26 chữ khác nhau, chữ kế tiếp chắc chắn lặp), Space `O(26) = O(1)`.

### Cách 3 — Mảng boolean 26 phần tử

Thay `set` bằng mảng `seen[26]`. Không cần băm; chỉ số suy từ `ord`.

In [13]:
from typing import List

class SolutionArray:
    def repeatedCharacter(self, s: str) -> str:
        seen = [False] * 26
        for ch in s:
            k = ord(ch) - ord("a")
            if seen[k]:
                return ch
            seen[k] = True
        return ""

print(SolutionArray().repeatedCharacter("abccbaacz"))
print(SolutionArray().repeatedCharacter("abcdd"))

c
d


**Độ phức tạp:** Time `O(n)`, Space `O(26) = O(1)`.

### Cách 4 — Bitmask 26 bit

Một số nguyên `mask` thay cho mảng: `mask >> k & 1` để hỏi, `mask |= 1 << k` để ghi. Gọn và rất nhanh.

In [14]:
from typing import List

class SolutionBitmask:
    def repeatedCharacter(self, s: str) -> str:
        mask = 0
        for ch in s:
            bit = 1 << (ord(ch) - ord("a"))
            if mask & bit:
                return ch
            mask |= bit
        return ""

print(SolutionBitmask().repeatedCharacter("abccbaacz"))
print(SolutionBitmask().repeatedCharacter("abcdd"))

c
d


**Độ phức tạp:** Time `O(n)`, Space `O(1)` (một số nguyên 26 bit).

### Cách 5 — Chỉ số lần xuất hiện thứ hai nhỏ nhất bằng `str.find`

Với mỗi chữ cái `c` có trong `s`: `i = s.find(c)`, `j = s.find(c, i + 1)`. Nếu `j != -1` thì `j` là vị trí lần thứ hai của `c`. Trả về chữ cái có `j` nhỏ nhất. Nghĩ theo "từng chữ cái" thay vì "từng vị trí".

In [15]:
import string
from typing import List

class SolutionSecondIndex:
    def repeatedCharacter(self, s: str) -> str:
        best_char, best_index = "", len(s)
        for c in string.ascii_lowercase:
            i = s.find(c)
            if i == -1:
                continue
            j = s.find(c, i + 1)
            if j != -1 and j < best_index:
                best_char, best_index = c, j
        return best_char

print(SolutionSecondIndex().repeatedCharacter("abccbaacz"))
print(SolutionSecondIndex().repeatedCharacter("abcdd"))

c
d


**Độ phức tạp:** 26 chữ cái × hai lần `find` `O(n)` → Time `O(26·n) = O(n)`, Space `O(1)`. Chậm hơn các cách trước nhưng thể hiện đúng định nghĩa "lần thứ hai sớm nhất".

## 8. TRACE từng bước

Trace Cách 4 (bitmask) với `s = "abccbaacz"`:

| chỉ số | `ch` | `mask` trước | bit đã bật? | hành động |
|---|---|---|---|---|
| 0 | `a` | `0` | không | bật bit `a` |
| 1 | `b` | `0b1` | không | bật bit `b` |
| 2 | `c` | `0b11` | không | bật bit `c` |
| 3 | `c` | `0b111` | **có** | trả về `c` |

In [16]:
def trace_bitmask(s):
    mask = 0
    print(f"s={s!r}")
    for i, ch in enumerate(s):
        bit = 1 << (ord(ch) - 97)
        already = bool(mask & bit)
        print(f"  i={i}: ch={ch!r} mask={mask:026b}"[:60] + f"  đã bật={already}")
        if already:
            print(f"  -> trả về {ch!r}")
            return ch
        mask |= bit
    return ""

print(trace_bitmask("abccbaacz"))
print(trace_bitmask("abcdd"))

s='abccbaacz'
  i=0: ch='a' mask=00000000000000000000000000  đã bật=False
  i=1: ch='b' mask=00000000000000000000000001  đã bật=False
  i=2: ch='c' mask=00000000000000000000000011  đã bật=False
  i=3: ch='c' mask=00000000000000000000000111  đã bật=True
  -> trả về 'c'
c
s='abcdd'
  i=0: ch='a' mask=00000000000000000000000000  đã bật=False
  i=1: ch='b' mask=00000000000000000000000001  đã bật=False
  i=2: ch='c' mask=00000000000000000000000011  đã bật=False
  i=3: ch='d' mask=00000000000000000000000111  đã bật=False
  i=4: ch='d' mask=00000000000000000000001111  đã bật=True
  -> trả về 'd'
d


In [17]:
def trace_set(s):
    seen = []
    print(f"s={s!r}")
    for i, ch in enumerate(s):
        if ch in seen:
            print(f"  i={i}: {ch!r} đã có trong {seen} -> dừng, trả về {ch!r}")
            return ch
        seen.append(ch)
        print(f"  i={i}: {ch!r} mới, seen={seen}")
    return ""

trace_set("zzabc")
trace_set("abcdefghijklmnopqrstuvwxyza")      # 26 chữ khác nhau rồi lặp 'a'

s='zzabc'
  i=0: 'z' mới, seen=['z']
  i=1: 'z' đã có trong ['z'] -> dừng, trả về 'z'
s='abcdefghijklmnopqrstuvwxyza'
  i=0: 'a' mới, seen=['a']
  i=1: 'b' mới, seen=['a', 'b']
  i=2: 'c' mới, seen=['a', 'b', 'c']
  i=3: 'd' mới, seen=['a', 'b', 'c', 'd']
  i=4: 'e' mới, seen=['a', 'b', 'c', 'd', 'e']
  i=5: 'f' mới, seen=['a', 'b', 'c', 'd', 'e', 'f']
  i=6: 'g' mới, seen=['a', 'b', 'c', 'd', 'e', 'f', 'g']
  i=7: 'h' mới, seen=['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h']
  i=8: 'i' mới, seen=['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i']
  i=9: 'j' mới, seen=['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j']
  i=10: 'k' mới, seen=['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k']
  i=11: 'l' mới, seen=['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l']
  i=12: 'm' mới, seen=['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm']
  i=13: 'n' mới, seen=['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n']
  i=14: 'o' mới, seen=['a', 'b', 'c', 'd', 'e

'a'

## 9. Code nộp LeetCode

In [18]:
from typing import List

class Solution:
    def repeatedCharacter(self, s: str) -> str:
        seen = set()
        for ch in s:
            if ch in seen:
                return ch
            seen.add(ch)
        return ""

print(Solution().repeatedCharacter("abccbaacz"))
print(Solution().repeatedCharacter("abcdd"))

c
d


## 10. Vì sao thuật toán đúng

- **Invariant:** sau khi xử lý các chỉ số `0..i-1`, `seen` chứa **đúng** các chữ cái xuất hiện trong `s[0..i-1]`, và chưa có chữ cái nào lặp trong đoạn đó (nếu có, ta đã trả về).
- **Phát hiện lần thứ hai:** tại chỉ số `i`, `s[i] ∈ seen` ⇔ `s[i]` đã xuất hiện ở một chỉ số `< i` ⇔ `i` là lần xuất hiện **thứ hai trở đi** của chữ đó. Vì trước `i` chưa có chữ nào lặp (invariant), `i` chính là **lần thứ hai đầu tiên**: nhỏ nhất trong mọi chữ cái.
- **Tính duy nhất:** chữ cái cần tìm là chữ có lần xuất hiện thứ hai ở chỉ số nhỏ nhất, tức đúng `s[i]` tại chỉ số `i` nhỏ nhất thỏa `s[i] ∈ s[:i]`.
- **Luôn dừng:** đề đảm bảo có chữ lặp; hơn nữa chỉ có 26 chữ cái, nên sau tối đa 27 ký tự chắc chắn có lặp (nguyên lý chuồng bồ câu).
- **Cách 5** tính trực tiếp `min` trên các chữ cái của vị trí lần thứ hai, cho cùng đáp án theo định nghĩa.

## 11. Phân tích Time / Space Complexity

`n = len(s) ≤ 100`, bảng chữ cái `26`.

| Cách | Time | Space |
|---|---|---|
| 1. Nhìn lại `s[:j]` | `O(n²)` | `O(n)` tạm |
| 2. `set` | `O(n)` (≤ 27 bước) | `O(26)` |
| 3. Mảng boolean 26 | `O(n)` | `O(26)` |
| 4. Bitmask | `O(n)` | `O(1)` |
| 5. `find` lần thứ hai | `O(26·n)` | `O(1)` |

Nhờ nguyên lý chuồng bồ câu, cách 2–4 thực tế chỉ lặp tối đa 27 lần; mọi giới hạn trên đều rất rộng.

## 12. Bộ test

Ví dụ đề, edge case (hai ký tự, lặp ngay đầu, lặp cuối cùng, 26 chữ khác nhau rồi lặp, nhiều chữ lặp nhưng chỉ một chữ lần thứ hai sớm nhất) và test ngẫu nhiên đối chiếu **tất cả cách** (chỉ sinh chuỗi **có chữ lặp**, đúng như đề đảm bảo).

In [19]:
import random, string

solvers = [SolutionBrute(), SolutionSet(), SolutionArray(), SolutionBitmask(), SolutionSecondIndex(), Solution()]

fixed = [
    ("abccbaacz", "c"),
    ("abcdd", "d"),
    ("aa", "a"),                                  # ngắn nhất
    ("aab", "a"),                                 # lặp ngay đầu
    ("abb", "b"),
    ("abcdefghijklmnopqrstuvwxyza", "a"),        # 26 khác nhau rồi lặp
    ("abcdefghijklmnopqrstuvwxyzz", "z"),
    ("abab", "a"),                                # a lặp ở chỉ số 2, b ở chỉ số 3
    ("baab", "a"),                                # a lặp ở 2 (trước b ở 3)
    ("abcabc", "a"),
    ("xyzzyx", "z"),
    ("qwertyqwerty", "q"),
    ("aabbcc", "a"),
    ("abcbcaa", "b"),                             # b lặp ở chỉ số 3, c ở 4
]
for s, expected in fixed:
    for sol in solvers:
        assert sol.repeatedCharacter(s) == expected, (type(sol).__name__, s, expected)
print("test cố định: OK")

def reference(s):
    # chữ cái có chỉ số lần thứ hai nhỏ nhất, dựa trên đếm trực tiếp
    count = {}
    for ch in s:
        count[ch] = count.get(ch, 0) + 1
        if count[ch] == 2:
            return ch

random.seed(2351)
generated = 0
while generated < 800:
    n = random.randint(2, 40)
    alphabet = random.choice(["ab", "abc", "abcdefgh", string.ascii_lowercase])
    s = "".join(random.choice(alphabet) for _ in range(n))
    if len(set(s)) == len(s):          # đề đảm bảo có chữ lặp
        continue
    generated += 1
    expected = reference(s)
    for sol in solvers:
        assert sol.repeatedCharacter(s) == expected, (type(sol).__name__, s)
print("800 test ngẫu nhiên: OK")

test cố định: OK
800 test ngẫu nhiên: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chọn chữ có **lần xuất hiện đầu** sớm nhất trong các chữ lặp

Đề hỏi lần xuất hiện **thứ hai** sớm nhất. Ở `"abccbaacz"`, chữ `a` có lần đầu ở 0 nhưng lần thứ hai ở 5; chữ `c` mới đúng.

In [20]:
def wrong1(s):
    repeated = [ch for ch in set(s) if s.count(ch) >= 2]
    return min(repeated, key=s.find)           # theo lần đầu

print("sai :", wrong1("abccbaacz"), "(đáng lẽ 'c')")
print("đúng:", Solution().repeatedCharacter("abccbaacz"))

sai : a (đáng lẽ 'c')
đúng: c


### Lỗi 2: Thêm vào `seen` **trước** khi kiểm tra

Khi đó mọi chữ đều "đã thấy" ngay ở lần đầu.

In [21]:
def wrong2(s):
    seen = set()
    for ch in s:
        seen.add(ch)                           # sai: thêm trước
        if ch in seen:
            return ch
    return ""

print("sai :", wrong2("abccbaacz"), "(đáng lẽ 'c')")
print("đúng:", Solution().repeatedCharacter("abccbaacz"))

sai : a (đáng lẽ 'c')
đúng: c


### Lỗi 3: Lấy chữ cái xuất hiện **nhiều nhất**

Đề không hỏi tần suất cao nhất.

In [22]:
from collections import Counter

def wrong3(s):
    return Counter(s).most_common(1)[0][0]

print("sai :", wrong3("abcdd" + "ee" + "e"), "| ví dụ 'abcddeee' đáng lẽ 'd'")
print("đúng:", Solution().repeatedCharacter("abcddeee"))

sai : e | ví dụ 'abcddeee' đáng lẽ 'd'
đúng: d


### Lỗi 4: Dùng `set(s)` rồi so độ dài — chỉ biết **có** lặp, không biết chữ nào

`len(set(s)) < len(s)` cho biết có lặp, nhưng không trả được chữ cái.

In [23]:
def wrong4(s):
    has_repeat = len(set(s)) < len(s)
    return has_repeat                              # trả bool, không phải chữ cái

print("sai :", wrong4("abccbaacz"), "(đáng lẽ 'c')")
print("đúng:", Solution().repeatedCharacter("abccbaacz"))

sai : True (đáng lẽ 'c')
đúng: c


### Lỗi 5: Bỏ qua `+ 1` khi tìm lần thứ hai bằng `find`

`s.find(c, i)` bắt đầu **tại** `i` nên trả lại đúng `i` (lần đầu), không phải lần thứ hai.

In [24]:
def wrong5(s):
    best_char, best_index = "", len(s)
    for c in "abcdefghijklmnopqrstuvwxyz":
        i = s.find(c)
        if i == -1:
            continue
        j = s.find(c, i)                           # thiếu + 1: j == i
        if j != -1 and j < best_index:
            best_char, best_index = c, j
    return best_char

print("sai :", wrong5("abccbaacz"), "(trả chữ có lần đầu sớm nhất: 'a'; đáng lẽ 'c')")
print("đúng:", Solution().repeatedCharacter("abccbaacz"))

sai : a (trả chữ có lần đầu sớm nhất: 'a'; đáng lẽ 'c')
đúng: c


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Phần tử đầu tiên **lặp lại**" → duyệt một lượt với `set`, **kiểm tra trước, thêm sau**, trả ngay khi gặp lại.
- Bảng chữ cái nhỏ (26) → có thể thay `set` bằng **mảng boolean** hoặc **bitmask**.
- Phân biệt: "lần xuất hiện đầu" ≠ "lần xuất hiện thứ hai". Đọc kỹ đề: *"the first letter to appear twice"* → xét vị trí lần **thứ hai**.
- Nguyên lý chuồng bồ câu: 27 chữ cái thì chắc chắn có lặp; vòng lặp tự dừng sớm.
- Cùng họ: LeetCode 217 Contains Duplicate, 387 First Unique Character, 2309, 136 Single Number.

## 15. Biến thể của bài

1. Trả về **chữ cái đầu tiên** xuất hiện **ba lần** (hoặc `k` lần): dùng `Counter` tích luỹ hoặc từ điển đếm.
2. LeetCode 217 Contains Duplicate: chỉ cần biết có trùng hay không.
3. Trả về **chỉ số** của lần xuất hiện thứ hai (hoặc cặp chỉ số lần đầu & lần hai).
4. LeetCode 387 First Unique Character in a String: chữ cái xuất hiện **đúng một lần**, đầu tiên.
5. Liệt kê **mọi** chữ cái lặp theo thứ tự lần xuất hiện thứ hai.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Nhìn lại `s[:j]` | kiểm tra phần đầu | `O(n²)` | `O(n)` | hiểu bài, `n` rất nhỏ |
| 2. `set` đã thấy | kiểm tra trước, thêm sau | `O(n)` | `O(26)` | **khuyến nghị khi nộp** |
| 3. Mảng boolean 26 | chỉ số `ord - 97` | `O(n)` | `O(26)` | tránh băm, bảng chữ cái nhỏ |
| 4. Bitmask | một số nguyên 26 bit | `O(n)` | `O(1)` | kỹ thuật bit, rất nhanh |
| 5. `find` lần thứ hai | min các vị trí lần hai | `O(26n)` | `O(1)` | theo định nghĩa từng chữ cái |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** `print("abc".find("c"), "abca".find("a", 1), "abc".find("z"), "a" in set("ab"))` in ra gì?

**Bài 2 (tính tay).** `s = "abcbca"`. Chữ nào xuất hiện lần thứ hai đầu tiên? Ở chỉ số bao nhiêu?

**Bài 3 (điền TODO).** Hoàn thành hàm trả về **chỉ số** của lần xuất hiện thứ hai sớm nhất (hoặc `-1`).

In [25]:
def first_repeat_index(s):
    # TODO: dùng set, trả về chỉ số i đầu tiên mà s[i] đã thấy trước đó
    pass

# chưa gọi hàm ở đây để notebook chạy không lỗi

**Bài 4 (tự viết lại).** Viết `first_unique(s)` (LeetCode 387): trả về **chỉ số** của chữ cái xuất hiện **đúng một lần**, đầu tiên; `-1` nếu không có.

In [26]:
def first_unique(s):
    # TODO: Counter rồi quét lại s
    pass

**Bài 5 (mini challenge).** Viết `first_k_times(s, k)` trả về chữ cái **đầu tiên đạt `k` lần xuất hiện** khi duyệt từ trái sang phải (hoặc `""` nếu không có). Với `k = 2` đây là bài gốc.

In [27]:
def first_k_times(s, k):
    # TODO: từ điển đếm tăng dần, trả về ngay khi count == k
    pass

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `find("c")=2`; `"abca".find("a", 1)=3`; `find("z")=-1`; `"a" in set("ab")` là `True`. Bài 2: duyệt `a,b,c` mới, rồi `b` đã có ở chỉ số 3.
- **Mức 2 (vừa):** Bài 3: giống cách 2 nhưng dùng `enumerate` để trả `i`. Bài 4: `Counter(s)` rồi `for i, ch in enumerate(s): if cnt[ch] == 1: return i`.
- **Mức 3 (sâu):** Bài 5: `count[ch] = count.get(ch, 0) + 1`; nếu `count[ch] == k` thì trả `ch`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
from collections import Counter

# Bài 1 và 2
assert ("abc".find("c"), "abca".find("a", 1), "abc".find("z"), "a" in set("ab")) == (2, 3, -1, True)
assert Solution().repeatedCharacter("abcbca") == "b"

# Bài 3
def first_repeat_index(s):
    seen = set()
    for i, ch in enumerate(s):
        if ch in seen:
            return i
        seen.add(ch)
    return -1

# Bài 4
def first_unique(s):
    cnt = Counter(s)
    for i, ch in enumerate(s):
        if cnt[ch] == 1:
            return i
    return -1

# Bài 5
def first_k_times(s, k):
    count = {}
    for ch in s:
        count[ch] = count.get(ch, 0) + 1
        if count[ch] == k:
            return ch
    return ""

assert first_repeat_index("abcbca") == 3
assert first_repeat_index("abc") == -1
assert first_repeat_index("aa") == 1
assert first_unique("leetcode") == 0
assert first_unique("loveleetcode") == 2
assert first_unique("aabb") == -1
assert first_k_times("abccbaacz", 2) == "c"
assert first_k_times("abcabcabc", 3) == "a"
assert first_k_times("abc", 2) == ""
assert first_k_times("aabbb", 3) == "b"
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (phần tử đầu tiên lặp lại)

```python
class Solution:
    def repeatedCharacter(self, s: str) -> str:
        seen = set()
        for ch in s:
            if ch in seen:      # kiểm tra TRƯỚC
                return ch
            seen.add(ch)        # thêm SAU
        return ""
```

**Checklist**

- [ ] Kiểm tra `ch in seen` **trước**, `seen.add(ch)` **sau**?
- [ ] Trả về ngay khi gặp lần thứ hai đầu tiên (đúng nghĩa "first to appear twice")?
- [ ] Không nhầm với chữ có lần xuất hiện **đầu** sớm nhất hoặc chữ nhiều nhất?
- [ ] Nếu dùng `find`, nhớ `+ 1` khi tìm lần thứ hai?
- [ ] Đã thử `"aa"`, lặp ngay đầu, 26 chữ khác nhau rồi lặp, và ví dụ `"baab"`/`"abab"`?
- [ ] Có `return ""` (hoặc đảm bảo đề luôn có đáp án) ở cuối hàm?
- [ ] Signature đúng `repeatedCharacter(self, s: str) -> str` và có `from typing import List`?